<a href="https://colab.research.google.com/github/IshitaSharma009/ML/blob/Feature-Engineering/pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle
!chmod 600 /root/.kaggle/kaggle.json

In [2]:
!kaggle datasets download shuofxz/titanic-machine-learning-from-disaster

Dataset URL: https://www.kaggle.com/datasets/shuofxz/titanic-machine-learning-from-disaster
License(s): DbCL-1.0
  0% 0.00/33.1k [00:00<?, ?B/s]
100% 33.1k/33.1k [00:00<00:00, 112MB/s]


In [3]:
import zipfile
zip_ref=zipfile.ZipFile('titanic-machine-learning-from-disaster.zip','r')
zip_ref.extractall('/content')
zip_ref.close()

In [4]:
import numpy as np
import pandas as pd

from sklearn.preprocessing import MinMaxScaler
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.feature_selection import SelectKBest, chi2
from sklearn.tree import DecisionTreeClassifier

In [5]:
df=pd.read_csv('train.csv',index_col='PassengerId')

In [6]:
df.head()

,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
PassengerId,,,,,,,,,,,
1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [7]:
df=df.drop(columns=['Name','Ticket','Cabin'])

In [8]:
df.head()

,Survived,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked
PassengerId,,,,,,,,
1,0,3,male,22.0,1,0,7.2500,S
2,1,1,female,38.0,1,0,71.2833,C
3,1,3,female,26.0,0,0,7.9250,S
4,1,1,female,35.0,1,0,53.1000,S
5,0,3,male,35.0,0,0,8.0500,S


In [9]:
x_train,x_test,y_train,y_test=train_test_split(df.drop(columns=['Survived']),df['Survived'],test_size=0.2,random_state=42)

In [10]:
x_test.info()

<class 'pandas.core.frame.DataFrame'>
Index: 179 entries, 710 to 11
Data columns (total 7 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   Pclass    179 non-null    int64  
 1   Sex       179 non-null    object 
 2   Age       142 non-null    float64
 3   SibSp     179 non-null    int64  
 4   Parch     179 non-null    int64  
 5   Fare      179 non-null    float64
 6   Embarked  179 non-null    object 
dtypes: float64(2), int64(3), object(2)
memory usage: 11.2+ KB


In [12]:
pipe_cat=make_pipeline([
    ('impute_embarked',SimpleImputer(strategy='most_frequent')),
    ('ohe_sex',OneHotEncoder(handle_unknown='ignore',sparse_output=False)),
    ('scaler',MinMaxScaler())
])

In [13]:
pipe_num=make_pipeline([
    ('impute_age',SimpleImputer()),
    ('scaler',MinMaxScaler())
])

In [29]:
trf1= ColumnTransformer([
    ('num',Pipeline([
    ('impute_age',SimpleImputer()),
    ('scaler',MinMaxScaler())
]),[2,5]),
    ('cat',Pipeline([
    ('impute_embarked',SimpleImputer(strategy='most_frequent')),
    ('ohe_sex',OneHotEncoder(handle_unknown='ignore',sparse_output=False)),
    ('scaler',MinMaxScaler())
]),[1,6])
],remainder='passthrough')

In [30]:
trf1.fit_transform(x_train)

array([[0.56647399, 0.0556283 , 0.        , ..., 1.        , 0.        ,
        0.        ],
       [0.28373963, 0.02537431, 0.        , ..., 2.        , 0.        ,
        0.        ],
       [0.39683338, 0.01546857, 0.        , ..., 3.        , 0.        ,
        0.        ],
       ...,
       [0.50992712, 0.02753757, 0.        , ..., 3.        , 2.        ,
        0.        ],
       [0.17064589, 0.2342244 , 1.        , ..., 1.        , 1.        ,
        2.        ],
       [0.25860769, 0.15085515, 0.        , ..., 1.        , 0.        ,
        1.        ]])

In [31]:
#feature selection
trf4=SelectKBest(score_func=chi2,k=8)

In [32]:
#model
trf5=DecisionTreeClassifier()

# **Pipeline**

In [33]:
pipe= make_pipeline(trf1,trf4,trf5)

In [34]:
pipe

Pipeline(steps=[('columntransformer',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('num',
                                                  Pipeline(steps=[('impute_age',
                                                                   SimpleImputer()),
                                                                  ('scaler',
                                                                   MinMaxScaler())]),
                                                  [2, 5]),
                                                 ('cat',
                                                  Pipeline(steps=[('impute_embarked',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('ohe_sex',
                                                                   OneHotEncoder(handle_unknown='ignore',
                                                                                 sparse_output=False)),
                                                                  ('scaler',
                                                                   MinMaxScaler())]),
                                                  [1, 6])])),
                ('selectkbest',
                 SelectKBest(k=8,
                             score_func=<function chi2 at 0x784aea93ede0>)),
                ('decisiontreeclassifier', DecisionTreeClassifier())])

In [35]:
pipe.fit(x_train, y_train)

Pipeline(steps=[('columntransformer',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('num',
                                                  Pipeline(steps=[('impute_age',
                                                                   SimpleImputer()),
                                                                  ('scaler',
                                                                   MinMaxScaler())]),
                                                  [2, 5]),
                                                 ('cat',
                                                  Pipeline(steps=[('impute_embarked',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('ohe_sex',
                                                                   OneHotEncoder(handle_unknown='ignore',
                                                                                 sparse_output=False)),
                                                                  ('scaler',
                                                                   MinMaxScaler())]),
                                                  [1, 6])])),
                ('selectkbest',
                 SelectKBest(k=8,
                             score_func=<function chi2 at 0x784aea93ede0>)),
                ('decisiontreeclassifier', DecisionTreeClassifier())])

In [36]:
y_pred=pipe.predict(x_test)

In [38]:
y_pred

array([0, 0, 0, 1, 1, 1, 1, 0, 1, 1, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0,
       0, 0, 1, 0, 0, 1, 1, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 1, 0, 0, 1,
       0, 0, 0, 1, 1, 1, 1, 1, 0, 0, 1, 1, 1, 1, 0, 1, 1, 0, 1, 0, 1, 1,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0, 1, 1, 0, 0, 1,
       1, 0, 1, 1, 0, 0, 1, 1, 0, 0, 1, 1, 1, 0, 0, 0, 1, 0, 0, 0, 1, 0,
       0, 1, 1, 0, 1, 0, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0, 0, 0,
       0, 0, 1, 1, 1, 1, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0, 1, 1, 0, 0, 0,
       0, 1, 1])

In [37]:
from sklearn.metrics import accuracy_score
accuracy_score(y_pred,y_test)

0.7877094972067039